> forward pass (next token logits) vs. decoding phase (token decision)

- 草稿小模型 (Draft Model) & 目标大模型 (Target Model)
- (draft model) tokens speculation
    - forward pass + auto-regressive decoding (token by token)
        - token decision: sampling strategy (argmax, top-k, top-p)
    - ar 是串行，串行生成
- (target model) **parallel** verification
    - forward pass: 1 次
    - 并行验证
- rejection sampling

-----

- 推测 (Speculation)：
    - 在当前文本序列 $x$ 的基础上，让草稿模型 M_draft 以自回归的方式快速生成 $K$ 个候选词元（tokens），形成一个草稿序列 $γ = (γ_1, γ_2, ..., γ_K)$。
例如，当前文本是“今天天气”，草稿模型可能快速生成了“真的很好，阳光明媚”。
- 验证 (Verification)：
    - 将原始文本 x 和整个草稿序列 γ 一次性输入给目标模型 M_target。
    - M_target 进行一次前向传播（Forward Pass）。这次计算会并行地输出在每个位置上，它自己“认为”的下一个词的概率分布。也就是说，M_target 会告诉你：
        - 在“今天天气”之后，它想生成的第一个词是什么。
        - 在“今天天气真的”之后，它想生成的第二个词是什么。
        - 在“今天天气真的很好”之后，它想生成的第三个词是什么。
        - ...依此类推。
    - 这里简单展开下（forward pass（已知 input tokens）的情况下是完全可以并行的）
        - 输入: `[T1, T2, T3, T4]`
        - 输出: `[Logits_1, Logits_2, Logits_3, Logits_4]`
        - 这里的关键是 因果注意力遮罩（Causal Attention Mask） 的作用。这个遮罩确保了在计算 Logits_i 时，模型只能“看到” 从 T1 到 T_i 的信息，而不能“偷看”未来的 T_{i+1} 等。
            - Logits_1 是基于 `[T1]` 计算出的，它代表了模型对 T2 的预测。
            - Logits_2 是基于 `[T1, T2]` 计算出的，它代表了模型对 T3 的预测。
            - Logits_3 是基于 `[T1, T2, T3]` 计算出的，它代表了模型对 T4 的预测。
            - Logits_4 是基于 `[T1, T2, T3, T4]` 计算出的，它代表了模型对 T5 的预测。
        - 在常规的自回归生成中，我们只关心最后一个 Logits_4，用它来采样生成 T5。然后把 T5 加入输入，再进行下一次完整的计算。我们把前面计算出的 Logits_1, Logits_2, Logits_3 都“浪费”掉了。
- 比较与接受 (Comparison & Acceptance)：
    - $DP$ (draft prob) vs. $TP$(target prob)
        - if tp >= dp, accepted
    - 从草稿的第一个词 γ_1 开始，逐一进行比较。
        - 比较 γ_1：M_target 在“今天天气”之后想生成的词，和草稿 γ_1（“真的”）是否一致？
            - 如果一致，则接受 γ_1，继续比较下一个。
        - 比较 γ_2：M_target 在“今天天气真的”之后想生成的词，和草稿 γ_2（“很好”）是否一致？
            - 如果一致，则接受 γ_2，继续比较。
        - ...直到出现不匹配：假设在比较 γ_3（“，”）时，M_target 认为在“今天天气真的很好”之后应该生成的是“！”。这时，不匹配发生了。
        - 处理不匹配：我们接受所有匹配的词元（γ_1, γ_2），然后拒绝不匹配的 γ_3 以及其后所有的草稿词元。
- 修正与迭代 (Correction & Iteration)：
模型最终的输出是所有被接受的词元，加上由 M_target 亲自计算出的那个不匹配位置的正确词元。
在上面的例子中，最终输出会增加“真的很好！”。
然后，从这个新的、被验证过的文本序列开始，重复第一步，让草稿模型再次进行推测。

In [ ]:
!python --version

- 上下文 (Prompt): `Why did the chicken` 草稿词元 (Draft Tokens, K=4): cross the farm ?

|  | `cross` | `the` | `farm` | `?` | `To` (额外生成) |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **DP** | 0.7 | 0.9 | 0.8 | 0.8 | - |
| **TP** | 0.9 | 0.9 | 0.7 | 0.9 | 0.8 |
||✔️|✔️|×|×|×||

- 最好的情况，一次 target model forward pass 得到 k+1 个 tokens
- 最坏的情况，一次 target model 的前向也能得到一个 token,即在当前第一个<DP的位置进行的一次推断；

### infra

In [ ]:
!pip install --upgrade vllm

- https://openlm.ai/speculative-decoding-in-vllm/

In [13]:
from vllm import LLM

llm = LLM(
    model="facebook/opt-6.7b",
    tensor_parallel_size=1,
    speculative_config={
        "method": "ngram",
        "num_speculative_tokens": 5,
        "prompt_lookup_max": 4,
    },
)
outputs = llm.generate("The future of AI is")

for output in outputs:
    print(f"Prompt: {output.prompt!r}, Generated text: {output.outputs[0].text!r}")

INFO 02-03 12:39:50 [utils.py:261] non-default args: {'disable_log_stats': True, 'speculative_config': {'method': 'ngram', 'num_speculative_tokens': 5, 'prompt_lookup_max': 4}, 'model': 'facebook/opt-6.7b'}
INFO 02-03 12:39:50 [model.py:541] Resolved architecture: OPTForCausalLM
INFO 02-03 12:39:50 [model.py:1561] Using max model len 2048
INFO 02-03 12:39:50 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=21087) INFO 02-03 12:39:50 [core.py:96] Initializing a V1 LLM engine (v0.15.0) with config: model='facebook/opt-6.7b', speculative_config=SpeculativeConfig(method='ngram', model=None, num_spec_tokens=5), tokenizer='facebook/opt-6.7b', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantizati

(EngineCore_DP0 pid=21087) /home/jiawei-wang/miniconda3/envs/rl/lib/python3.10/site-packages/tvm_ffi/_optional_torch_c_dlpack.py:174: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
(EngineCore_DP0 pid=21087) We recommend installing via `pip install torch-c-dlpack-ext`
(EngineCore_DP0 pid=21087)   warnings.warn(


(EngineCore_DP0 pid=21087) INFO 02-03 12:39:52 [cuda.py:364] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')


pytorch_model-00001-of-00002.bin:   0%|          | 0.00/9.96G [00:00<?, ?B/s]

pytorch_model-00002-of-00002.bin:   0%|          | 0.00/3.36G [00:00<?, ?B/s]

(EngineCore_DP0 pid=21087) INFO 02-03 12:43:29 [weight_utils.py:527] Time spent downloading weights for facebook/opt-6.7b: 216.773026 seconds


Loading pt checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(EngineCore_DP0 pid=21087) INFO 02-03 12:43:35 [default_loader.py:291] Loading weights took 5.90 seconds
(EngineCore_DP0 pid=21087) INFO 02-03 12:43:35 [gpu_model_runner.py:4048] Loading drafter model...
(EngineCore_DP0 pid=21087) INFO 02-03 12:43:36 [gpu_model_runner.py:4118] Model loading took 12.4 GiB memory and 224.018016 seconds
(EngineCore_DP0 pid=21087) INFO 02-03 12:43:38 [backends.py:805] Using cache directory: /home/jiawei-wang/.cache/vllm/torch_compile_cache/086d07fc1d/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=21087) INFO 02-03 12:43:38 [backends.py:865] Dynamo bytecode transform time: 1.86 s
(EngineCore_DP0 pid=21087) ERROR 02-03 12:43:39 [core.py:946] EngineCore failed to start.
(EngineCore_DP0 pid=21087) ERROR 02-03 12:43:39 [core.py:946] Traceback (most recent call last):
(EngineCore_DP0 pid=21087) ERROR 02-03 12:43:39 [core.py:946]   File "/home/jiawei-wang/miniconda3/envs/rl/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 937, in run_eng

(EngineCore_DP0 pid=21087) Process EngineCore_DP0:
(EngineCore_DP0 pid=21087) Traceback (most recent call last):
(EngineCore_DP0 pid=21087)   File "/home/jiawei-wang/miniconda3/envs/rl/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore_DP0 pid=21087)     self.run()
(EngineCore_DP0 pid=21087)   File "/home/jiawei-wang/miniconda3/envs/rl/lib/python3.10/multiprocessing/process.py", line 108, in run
(EngineCore_DP0 pid=21087)     self._target(*self._args, **self._kwargs)
(EngineCore_DP0 pid=21087)   File "/home/jiawei-wang/miniconda3/envs/rl/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 950, in run_engine_core
(EngineCore_DP0 pid=21087)     raise e
(EngineCore_DP0 pid=21087)   File "/home/jiawei-wang/miniconda3/envs/rl/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 937, in run_engine_core
(EngineCore_DP0 pid=21087)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore_DP0 pid=21087)   File "/home/jiawei-wan

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}